# Extract Alibaba Stress-Test Windows

Extracts 4 specific step-range windows from the Alibaba 7-day trace.
Run this on Kaggle (where the raw trace is available) and download the output  files.

In [ ]:
# Cell 1: Setup paths
import os, sys, glob
import numpy as np

_candidates = [
    "/kaggle/input/alibaba-cluster-data",
    "/kaggle/input/datasets/rohitdurbha/alibaba-cluster-data",
]
ALIBABA_INPUT = next((p for p in _candidates if os.path.isdir(p)), None)
if ALIBABA_INPUT is None:
    hits = glob.glob("/kaggle/input/**/machine_usage*.csv", recursive=True)
    if hits:
        ALIBABA_INPUT = os.path.dirname(sorted(hits, key=os.path.getsize, reverse=True)[0])

PROJECT_DIR = "/kaggle/input/datasets/autocloud-agent" if os.path.isdir("/kaggle/input/datasets/autocloud-agent") else "/kaggle/working"
sys.path.insert(0, PROJECT_DIR)

OUT_DIR = "/kaggle/working/stress_windows"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Alibaba input: {ALIBABA_INPUT}")
print(f"Output dir   : {OUT_DIR}")


In [ ]:
# Cell 2: Load full 7-day trace
from environment.workload import AlibabaTraceLoader

loader = AlibabaTraceLoader(
    data_dir=ALIBABA_INPUT,
    n_machines=5000,
    bin_size_s=30,
)
loader.load(verbose=True, chunk_size=500_000)

full_data    = loader.get_full_data()   # (N_total_bins, 4)
bins_per_day = int(86400 / 30)          # 2880 bins per day
n_days       = len(full_data) // bins_per_day
print(f"Full trace: {len(full_data)} bins = {n_days} days")

days = [full_data[d*bins_per_day : (d+1)*bins_per_day] for d in range(n_days)]
for i, d in enumerate(days):
    cpu = d[:, 0]
    print(f"  Day {i+1}: mean={cpu.mean():.3f}  max={cpu.max():.3f}  min={cpu.min():.3f}")


In [ ]:
# Cell 3: Extract stress windows

def save_window(arr, name):
    path = os.path.join(OUT_DIR, name)
    np.save(path, arr)
    cpu = arr[:, 0]
    print(f"Saved {name:40s}  shape={arr.shape}  cpu mean={cpu.mean():.3f}  max={cpu.max():.3f}")

# --- Scenario 1: Standard Ramp-Up (Day 2, steps 600-1000) ---
save_window(days[1][600:1000],    "s1_rampup_day2_600_1000.npy")

# --- Scenario 2: Early Shock (Day 5, steps 0-400) ---
if n_days >= 5:
    save_window(days[4][0:400],   "s2_earlyshock_day5_0_400.npy")
if n_days >= 6:
    save_window(days[5][0:400],   "s2_earlyshock_day6_0_400.npy")

# --- Scenario 3: Choppy Plateau (Day 3, steps 150-800) ---
if n_days >= 3:
    save_window(days[2][150:800], "s3_choppy_day3_150_800.npy")
if n_days >= 4:
    save_window(days[3][150:800], "s3_choppy_day4_150_800.npy")

# --- Scenario 4: Deep Trough + Recovery (Day 7, steps 1500-2880) ---
if n_days >= 7:
    save_window(days[6][1500:],   "s4_trough_day7_1500_2880.npy")

print("
All windows saved to", OUT_DIR)
print("Files:", os.listdir(OUT_DIR))


## Done

Download all files from  and place them in your local  folder.

Then run locally:
